# transcribe-colab: Google Colab GPU 高速文字起こし

Google Drive 上の音声・動画ファイル（`.mp3` / `.m4a` / `.wav` / `.mp4`）を、
`faster-whisper`（GPU `float16` 優先 / 失敗時は `int8` 自動フォールバック）で文字起こしします。

- 出力先: 各入力ファイルと同じ階層の `transcribed/` フォルダ（`<元ファイル名>.txt` と `<元ファイル名>.md`）
- 事前確認: メニュー「**ランタイム ＞ ランタイムのタイプを変更 ＞ T4 GPU**」が選択されていることを確認してから、上から順にセルを実行してください。


## 1. 設定（入力フォルダ・言語・モデル）

In [ ]:
#@title 文字起こし設定 { display-mode: "form" }
INPUT_DIR = "/content/drive/MyDrive/audio"  #@param {type:"string"}
LANGUAGE = "ja"  #@param ["ja", "en", "auto"]
MODEL = "auto"  #@param {type:"string"}
OVERWRITE = False  #@param {type:"boolean"}

# Colab 上で直接このノートブックを開いた場合に src/transcribe.py を取得するリポジトリ URL
REPO_URL = "https://github.com/imim100913-cell/transcribe-colab.git"


## 2. Google Drive のマウントと GPU 確認

In [ ]:
import subprocess

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("[INFO] Google Colab 環境外のため Drive マウントをスキップしました。")

res = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if res.returncode == 0 and res.stdout.strip():
    print(f"[INFO] GPU 検出: {res.stdout.strip()}")
else:
    print("[WARN] GPU が検出されません。Colab の『ランタイム > ランタイムのタイプを変更 > T4 GPU』を確認してください。")


## 3. 依存関係のインストールと本体コード (`src/transcribe.py`) の準備

In [ ]:
import subprocess
import sys
from pathlib import Path

# 依存パッケージをインストール
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faster-whisper>=1.1.0"])

# カレントディレクトリに src/transcribe.py が無ければ GitHub からクローン
local_repo = Path.cwd()
if not (local_repo / "src" / "transcribe.py").is_file():
    colab_repo = Path("/content/transcribe-colab")
    if colab_repo.exists():
        subprocess.run(["git", "-C", str(colab_repo), "pull", "--ff-only"], check=False)
    else:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(colab_repo)])
    repo_root = colab_repo
else:
    repo_root = local_repo

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import ctranslate2
print(f"[INFO] repo_root = {repo_root}")
print(f"[INFO] CUDA device count = {ctranslate2.get_cuda_device_count()}")


## 4. 文字起こしの実行

In [ ]:
from src.transcribe import transcribe_directory

summary = transcribe_directory(
    input_dir=INPUT_DIR,
    language=LANGUAGE,
    model=MODEL,
    overwrite=OVERWRITE,
)


## 5. 結果確認（サマリーと先頭出力のプレビュー）

In [ ]:
print("=== 実行結果サマリー ===")
print(f"入力フォルダ : {summary.input_dir}")
print(f"使用モデル   : {summary.model_name}")
print(f"デバイス     : {summary.device} ({summary.compute_type})")
print(
    f"件数         : total={summary.total_files}, "
    f"processed={summary.processed_count}, "
    f"skipped={summary.skipped_count}, "
    f"failed={summary.failed_count}"
)
print(f"総処理時間   : {summary.elapsed_seconds:.1f}s")

failures = [r for r in summary.results if r.status == "failed"]
if failures:
    print("\n[ERROR] 失敗したファイル:")
    for r in failures:
        print(f"  - {r.audio_path}: {r.error}")

preview_targets = [r for r in summary.results if r.status == "processed"] or [
    r for r in summary.results if r.status == "skipped" and r.md_path.is_file()
]
if preview_targets:
    first = preview_targets[0]
    print(f"\n--- プレビュー (.md): {first.md_path} ---")
    print("\n".join(first.md_path.read_text(encoding="utf-8").splitlines()[:20]))
    print(f"\n--- プレビュー (.txt): {first.txt_path} ---")
    print("\n".join(first.txt_path.read_text(encoding="utf-8").splitlines()[:10]))
